<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F03_combo_and_ensemble.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. The Terraform-deployed templates already carry the `SF_*` run identity in their env,
> so there's no environment cell to fill in. Runs on the **`sf-main`** runtime template (Python 3.11). See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.


# 03 · Combo + ensembles (one run_id)

The payoff. One config mixes a **Spark** model (`theta`) with the **BigQuery-native** models, and turns **ensembles on**. `main.run(cfg)` runs both compute tracks **in parallel under one `run_id`**, then — after both engines join — fires the **ensembler**: it blends the base forecasts (mean / median / inverse-error), scores each consensus against held-out actuals, and writes the results so they land on the *same* leaderboard as the base models.

One `v_model_leaderboard` query then shows **base Spark + base BigQuery + `ensemble_*`** side by side.

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable.

In [ ]:
# Cloud bootstrap: clone + editable-install so `import scale_forecasting` resolves.
# Harmless locally — if the package already imports, we do nothing.
import importlib.util
import os
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR], check=True)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses, so a notebook run and a Composer run land in the same registry. Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default. This demo targets the live `run_registry` + `v_model_leaderboard` / `v_run_summary`.

In [ ]:
from google.cloud import bigquery

from scale_forecasting.settings import Settings

settings = Settings.resolve()
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref
print("deployment:", DATASET, "region:", settings.region)

## Review helpers

Registry rows written through the Storage Write API are *async-visible*, so we poll the leaderboard briefly until a run's models show up. `leaderboard(run_id)` returns one row per model — `compute_engine` splits Spark / BigQuery / ensemble — and `run_summary(run_id)` is the header roll-up.

In [ ]:
import time

import pandas as pd


def _query(sql, run_id):
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", run_id)]
        ),
    )
    return job.result().to_dataframe()


def leaderboard(run_id, expect_models=None, tries=12, pause=3.0):
    """Poll v_model_leaderboard until `expect_models` all appear (or give up), newest metrics."""
    sql = (
        f"SELECT model_type, compute_engine, n_cells, mean_wape, mean_mae "
        f"FROM `{DATASET}.v_model_leaderboard` WHERE run_id=@run_id "
        f"ORDER BY mean_wape"
    )
    df = pd.DataFrame()
    for _ in range(tries):
        df = _query(sql, run_id)
        if expect_models is None or set(df["model_type"]) >= set(expect_models):
            break
        time.sleep(pause)
    return df


def run_summary(run_id):
    sql = f"SELECT * FROM `{DATASET}.v_run_summary` WHERE run_id=@run_id"
    return _query(sql, run_id)

## Parameters — edit me

Everything that shapes the run is set here as plain Python, then assembled into a **`RunConfig`** — the one frozen object that drives the run and is logged verbatim to `run_registry.raw_config`, so *this cell is the experiment record*. No separate JSON file to open.

- **`RUN_NAME`** carries a timestamp so each execution is its own clean run (the cell tables are append-only, so a fresh `run_id` avoids overwriting a still-buffering prior run).
- **`SOURCE_TABLE`** picks the storage format of the shipped input — `source_series_iceberg` (managed Apache Iceberg) or `source_series_native` (native BigQuery), identical series. Both the Spark and BigQuery tracks read whichever you name through the same table interface.
- **`MODELS`** mixes runtimes automatically: `theta` runs on **Spark** (`compute_engine='spark'`) while `arima_plus`/`timesfm` run in **BigQuery** — the router splits them by model.
- **`BACKTEST=True`** so the Spark path emits an OOF metric panel comparable to the natives; **`ENSEMBLE_STRATEGIES`** triggers the ensembler after the engine join (blends the base forecasts and scores each consensus).

In [ ]:
from scale_forecasting import main
from scale_forecasting.config import RunConfig
from scale_forecasting.registry.ids import make_run_id

# === Parameters — edit me ===============================================
RUN_NAME = f"nb03 combo ensemble {int(time.time())}"  # timestamped → a fresh run each execution
SOURCE_TABLE = "source_series_iceberg"  # shipped seed (100k); _iceberg↔_native to compare storage
MODELS = ["theta", "arima_plus", "timesfm"]  # theta → Spark; arima_plus/timesfm → BigQuery
HORIZON = 28
SERIES_LIMIT = 10  # small subset so the demo is quick + cheap
HOLIDAYS = ["US"]
BACKTEST = True  # OOF panel so ensembles can be scored/learned
N_FOLDS = 2
ENSEMBLE_STRATEGIES = ["mean", "median", "inverse_error"]
# ========================================================================

cfg = RunConfig(
    run_name=RUN_NAME,
    python_runtime="spark",
    data={"source_table": SOURCE_TABLE, "horizon": HORIZON, "series_limit": SERIES_LIMIT},
    models=MODELS,  # router splits Spark vs BigQuery automatically
    features={"holidays": HOLIDAYS},
    backtest={"enabled": BACKTEST, "n_folds": N_FOLDS, "horizon": HORIZON, "step": HORIZON},
    ensemble={"enabled": True, "strategies": ENSEMBLE_STRATEGIES},
)
run_id = make_run_id(cfg)
print("run_id:", run_id)
print("  models:", cfg.models)
print("  ensemble strategies:", cfg.ensemble.strategies)

returned = main.run(cfg)
assert returned == run_id
print("combo + ensemble run complete:", run_id)

## Review — base models and ensembles together

Base `theta` (`compute_engine='spark'`), the natives (`bigquery`), and each `ensemble_*` (`ensemble`) all on one board, ordered by `mean_wape`. If an ensemble beats every base model, it sits at the top.

In [ ]:
expect = list(cfg.models) + [f"ensemble_{s}" for s in cfg.ensemble.strategies]
board = leaderboard(run_id, expect_models=expect)
board

## The showpiece chart

`mean_wape` per model, colored by compute engine — base vs ensemble at a glance (lower is better).

In [ ]:
import matplotlib.pyplot as plt

plot_df = board.dropna(subset=["mean_wape"]).sort_values("mean_wape")
colors = {"spark": "#1f77b4", "bigquery": "#ff7f0e", "ensemble": "#2ca02c"}
bar_colors = [colors.get(e, "#888888") for e in plot_df["compute_engine"]]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(plot_df["model_type"], plot_df["mean_wape"], color=bar_colors)
ax.set_ylabel("mean WAPE (lower is better)")
ax.set_title(f"base vs ensemble — run {run_id[:12]}")
ax.tick_params(axis="x", rotation=45)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
ax.legend(handles, colors.keys(), title="compute_engine")
plt.tight_layout()
plt.show()

In [ ]:
run_summary(run_id)